In [3]:
import geopandas as gpd
import pandas as pd

In [19]:
gdf_nodes = gpd.GeoDataFrame.from_file("../map/data/nodes.geojson")
gdf_nodes['lon'] = gdf_nodes.geometry.apply(lambda geom: geom.x)
gdf_nodes['lat'] = gdf_nodes.geometry.apply(lambda geom: geom.y)
    

df_from_to = pd.read_csv("../map/data/flows.csv")


In [20]:

df_from_to

,from_id,to_id,flow_type,value,unit
0,WIND-01,BATTERY-01,Electricity,NaN,kW
1,SOLAR-01,BATTERY-01,Electricity,NaN,kW
2,BATTERY-01,ELEC-01,Electricity,NaN,kW
3,ELEC-01,WWTP-01,O2,NaN,m3/h
4,WWTP-01,ELEC-01,H2O,NaN,m3/h
5,ELEC-01,OFFICE-01,Waste Heat,NaN,kWh
6,ELEC-01,CREM-01,H2O,NaN,Kg/day


In [21]:
src = gdf_nodes.rename(columns={"id": "from_id", "lon": "from_lon", "lat": "from_lat"})[["from_id", "from_lon", "from_lat"]]
dst = gdf_nodes.rename(columns={"id": "to_id",   "lon": "to_lon",   "lat": "to_lat"})[["to_id",   "to_lon",   "to_lat"]]

In [22]:
src

,from_id,from_lon,from_lat
0,BATTERY-01,6.648297,52.366023
1,SOLAR-01,6.649537,52.364763
2,ELEC-01,6.648486,52.366094
3,WIND-01,6.648364,52.365398
4,OFFICE-01,6.650037,52.364847
5,WWTP-01,6.650514,52.366150
6,AADORP-20,6.626574,52.378455
7,AADORP-31,6.628404,52.376993
8,AADORP-46,6.631973,52.374302
9,CREM-01,6.830433,52.192469


In [23]:
dst

,to_id,to_lon,to_lat
0,BATTERY-01,6.648297,52.366023
1,SOLAR-01,6.649537,52.364763
2,ELEC-01,6.648486,52.366094
3,WIND-01,6.648364,52.365398
4,OFFICE-01,6.650037,52.364847
5,WWTP-01,6.650514,52.366150
6,AADORP-20,6.626574,52.378455
7,AADORP-31,6.628404,52.376993
8,AADORP-46,6.631973,52.374302
9,CREM-01,6.830433,52.192469


In [27]:
df_from_to = df_from_to.merge(src, on="from_id", how="left").merge(dst, on="to_id", how="left")

In [28]:
df_from_to

,from_id,to_id,flow_type,value,unit,from_lon,from_lat,to_lon,to_lat
0,WIND-01,BATTERY-01,Electricity,NaN,kW,6.648364,52.365398,6.648297,52.366023
1,WIND-01,BATTERY-01,Electricity,NaN,kW,6.648364,52.365398,6.648297,52.366023
2,SOLAR-01,BATTERY-01,Electricity,NaN,kW,6.649537,52.364763,6.648297,52.366023
3,BATTERY-01,ELEC-01,Electricity,NaN,kW,6.648297,52.366023,6.648486,52.366094
4,ELEC-01,WWTP-01,O2,NaN,m3/h,6.648486,52.366094,6.650514,52.366150
5,ELEC-01,WWTP-01,O2,NaN,m3/h,6.648486,52.366094,6.650514,52.366150
6,WWTP-01,ELEC-01,H2O,NaN,m3/h,6.650514,52.366150,6.648486,52.366094
7,WWTP-01,ELEC-01,H2O,NaN,m3/h,6.650514,52.366150,6.648486,52.366094
8,ELEC-01,OFFICE-01,Waste Heat,NaN,kWh,6.648486,52.366094,6.650037,52.364847
9,ELEC-01,CREM-01,H2O,NaN,Kg/day,6.648486,52.366094,6.830433,52.192469


In [33]:
df_from_to = df_from_to.drop_duplicates(subset=["from_id", "to_id", "flow_type"])


In [34]:
df_from_to

,from_id,to_id,flow_type,value,unit,from_lon,from_lat,to_lon,to_lat
0,WIND-01,BATTERY-01,Electricity,NaN,kW,6.648364,52.365398,6.648297,52.366023
2,SOLAR-01,BATTERY-01,Electricity,NaN,kW,6.649537,52.364763,6.648297,52.366023
3,BATTERY-01,ELEC-01,Electricity,NaN,kW,6.648297,52.366023,6.648486,52.366094
4,ELEC-01,WWTP-01,O2,NaN,m3/h,6.648486,52.366094,6.650514,52.366150
6,WWTP-01,ELEC-01,H2O,NaN,m3/h,6.650514,52.366150,6.648486,52.366094
8,ELEC-01,OFFICE-01,Waste Heat,NaN,kWh,6.648486,52.366094,6.650037,52.364847
9,ELEC-01,CREM-01,H2O,NaN,Kg/day,6.648486,52.366094,6.830433,52.192469


In [ ]:
COLOR_BY_TYPE = {
    "H2": [8, 104, 172],
    "O2": [102, 187, 106],
    "H2O": [38, 166, 154],
    "Heat": [251, 140, 0],
    "Electricity": [142, 36, 170]
}

In [25]:
from shapely.geometry import LineString
import random


# df_from_to["color"] = df_from_to["flow_type"].map(COLOR_BY_TYPE)

    # # Build a concise tooltip per flow (adjust fields if your CSV differs)
    # df_flows["tooltip"] = (
    #         df_flows["flow_type"].astype(str) + ": " +
    #         df_flows["from_id"].astype(str) + " → " + df_flows["to_id"].astype(str) +
    #         np.where(df_flows.get("value").notna(), " | " + df_flows["value"].astype(str), "") +
    #         np.where(df_flows.get("unit").notna(), " " + df_flows["unit"].astype(str), "")
    # )

    # --- Create LineString geometry for each edge ---
gdf_edges = gpd.GeoDataFrame(
    df_from_to,
    geometry=df_from_to.apply(
        lambda r: LineString([(r["from_lon"], r["from_lat"]), (r["to_lon"], r["to_lat"])]),
        axis=1,
    ),
    crs="EPSG:4326",
)



KeyError: 'from_lon'

In [11]:
gdf_edges

,from_id,to_id,flow_type,value,unit,from_lon,from_lat,to_lon,to_lat,geometry
0,WIND-01,BATTERY-01,Electricity,NaN,kW,6.648364,52.365398,6.648297,52.366023,"LINESTRING (6.64836 52.3654, 6.6483 52.36602)"
1,WIND-01,BATTERY-01,Electricity,NaN,kW,6.648364,52.365398,6.648297,52.366023,"LINESTRING (6.64836 52.3654, 6.6483 52.36602)"
2,SOLAR-01,BATTERY-01,Electricity,NaN,kW,6.649537,52.364763,6.648297,52.366023,"LINESTRING (6.64954 52.36476, 6.6483 52.36602)"
3,BATTERY-01,ELEC-01,Electricity,NaN,kW,6.648297,52.366023,6.648486,52.366094,"LINESTRING (6.6483 52.36602, 6.64849 52.36609)"
4,ELEC-01,WWTP-01,O2,NaN,m3/h,6.648486,52.366094,6.650514,52.366150,"LINESTRING (6.64849 52.36609, 6.65051 52.36615)"
5,ELEC-01,WWTP-01,O2,NaN,m3/h,6.648486,52.366094,6.650514,52.366150,"LINESTRING (6.64849 52.36609, 6.65051 52.36615)"
6,WWTP-01,ELEC-01,H2O,NaN,m3/h,6.650514,52.366150,6.648486,52.366094,"LINESTRING (6.65051 52.36615, 6.64849 52.36609)"
7,WWTP-01,ELEC-01,H2O,NaN,m3/h,6.650514,52.366150,6.648486,52.366094,"LINESTRING (6.65051 52.36615, 6.64849 52.36609)"
8,ELEC-01,OFFICE-01,Waste Heat,NaN,kWh,6.648486,52.366094,6.650037,52.364847,"LINESTRING (6.64849 52.36609, 6.65004 52.36485)"
9,ELEC-01,CREM-01,H2O,NaN,Kg/day,6.648486,52.366094,6.830433,52.192469,"LINESTRING (6.64849 52.36609, 6.83043 52.19247)"


In [12]:
with open("./data_constants.csv","r") as d_reader:
  DATA = pd.read_csv(d_reader, index_col=0)


# System definition and datasets used in this prototype.
ELECTROLYZER_KW = DATA.loc["el_rated_power_kw"]["value"]  


In [14]:
ELECTROLYZER_KW

np.float64(80.0)

In [ ]:
import streamlit as st

main_col, right_col = st.columns([6, 2], gap="small", vertical_alignment="top")

main_col

2025-11-20 16:59:11.735 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-20 16:59:11.736 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-20 16:59:11.737 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()